# Egoblind Dataset Evaluation

This notebook demonstrates how to load run outputs from the evaluation pipeline and visualize them.
It assumes you have already run the pipeline using:

```bash
python -m app.main --mode dataset_eval --dataset egoblind
```

In [ ]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Find the latest dataset_eval run
run_dirs = glob.glob("outputs/dataset_eval_*")
run_dirs.sort(key=os.path.getmtime, reverse=True)

if not run_dirs:
    print("No dataset evaluation runs found. Please run the pipeline first.")
else:
    latest_run_dir = run_dirs[0]
    print(f"Loading metrics from {latest_run_dir}")
    
    metrics_csv = os.path.join(latest_run_dir, "frame_metrics.csv")
    df = pd.read_csv(metrics_csv)
    print(f"Loaded {len(df)} frames.")

## 1. Latency over Time

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['frame_idx'], df['frame_total_latency_ms'], label='Total Frame Latency', color='black')
plt.plot(df['frame_idx'], df['yolo_latency_ms'], label='YOLO', alpha=0.7)
plt.plot(df['frame_idx'], df['depth_latency_ms'], label='Depth', alpha=0.7)
plt.title('Pipeline Latency over Time')
plt.xlabel('Frame Index')
plt.ylabel('Latency (ms)')
plt.legend()
plt.grid(True)
plt.show()

## 2. Component Latency Distribution

In [ ]:
latency_cols = ['yolo_latency_ms', 'depth_latency_ms', 'fusion_latency_ms', 'navigation_latency_ms', 'visualization_latency_ms']
available_cols = [c for c in latency_cols if c in df.columns]

plt.figure(figsize=(10, 6))
sns.boxplot(data=df[available_cols])
plt.title('Component Latency Distribution')
plt.ylabel('Latency (ms)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Instantaneous FPS

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['frame_idx'], df['fps_instant'], color='green')
plt.title('Instantaneous FPS over Time')
plt.xlabel('Frame Index')
plt.ylabel('FPS')
plt.grid(True)
plt.show()

## 4. Command Distribution

In [ ]:
if 'nav_command' in df.columns:
    counts = df['nav_command'].fillna('None').value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(counts, labels=counts.index, autopct='%1.1f%%')
    plt.title('Navigation Command Distribution')
    plt.show()